# Sweep di temperatura con prompt di **un solo token**

Ablazione controllata del protocollo del Capitolo 3 della tesi: **cambia solo il
prompt**, tutto il resto resta identico.

| | tesi (Cap. 4) | questo notebook |
|---|---|---|
| prompt | 2000 token da *Guerra e pace* | **un singolo token casuale** |
| modello | `SamueleZurlo/Qwen2.5-3B-v3` | lo stesso |
| temperature | 0.4 → 1.5, passo 0.1 | le stesse |
| campionamento | `top_p=1`, `top_k=-1`, `repetition_penalty=1` | lo stesso |
| lunghezza | 24 000 token generati | la stessa |
| analisi | 20 000 token `cl100k_base` | la stessa |

## Che cosa si sta verificando

Il §4.3 della tesi osserva che il descrittore $R$ cresce in modo **monotono** su
tutto l'intervallo e non presenta il massimo attorno a $T=1$ che Mikhaylovskiy
(*Zipf's and Heaps' Laws for Tokens and LLM-generated Texts*, Findings EMNLP
2025) riporta per tutti i modelli Qwen. La spiegazione data è che quel lavoro
genera **da un singolo token**, mentre la tesi usa un prompt lungo, e il §4.5
dello stesso paper dichiara che il prompt lungo

> «leads some models to generate no new tokens at all at low temperatures and
> generally shifts the maxima of $R$–$t$ plots to the high-temperature area».

Finora quella spiegazione è **citata, non misurata**. Questo notebook la misura.

## Condizioni del paper, §4.1

> «All the texts are generated from a single random, seed-controlled token.»
> «We generate texts at least 24K tokens long […] in a single run to fit the
> text generated into the context window.»
> «We do not use top-k, top-p or any other decoding parameters such as
> no-repeat to keep the things clean.»

## Tre scelte di disegno, e perché

**Token appaiato fra temperature.** Il paper media su 40–50 testi per annullare
l'effetto del token iniziale. Qui i campioni sono pochi, quindi il round $j$ usa
**lo stesso token a tutte e dodici le temperature**: il confronto fra temperature
diventa appaiato e il token sparisce come sorgente di varianza fra di esse.

**Ordine a giri, non a temperatura.** Il giro 1 genera un campione a ogni
temperatura da 0.4 a 1.5, poi il giro 2 ricomincia da capo. Interrompendo in
qualunque momento fra un giro e l'altro si ha copertura completa dello spettro e
lo stesso numero di campioni ovunque. È anche il rimedio al buco del dataset
vecchio, che a $T=1.0$ ha un solo documento e a $T=0.9$ tre.

**Niente `logit_bias` sull'EOS.** Sopprimere l'EOS con un bias negativo
azzererebbe le riprese e farebbe risparmiare, ma è a tutti gli effetti un
secondo parametro di decoding: violerebbe sia il «to keep the things clean» del
paper sia il §3.2 della tesi. L'EOS si gestisce per **continuazione**, come nel
protocollo originale. Per $R$ questo non costa nulla: è una misura di soli
conteggi, insensibile all'ordine, quindi le giunzioni non la toccano nemmeno in
linea di principio.

In [ ]:
# =====================================================================
# 0. SETUP  (Colab)
# =====================================================================
import subprocess, sys, importlib

def _assicura(pacchetto, modulo=None):
    try:
        importlib.import_module(modulo or pacchetto)
    except ImportError:
        print(f'installo {pacchetto} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pacchetto], check=True)

for p, m in [('openai', 'openai'), ('tiktoken', 'tiktoken'), ('transformers', 'transformers')]:
    _assicura(p, m)

import os, re, json, time, math, random, hashlib, threading, collections, datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import matplotlib.pyplot as plt
from openai import OpenAI
import tiktoken
from transformers import AutoTokenizer

try:
    display
except NameError:
    def display(x): print(x)

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.25,
    'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False, 'figure.figsize': (7.0, 4.4),
})

IN_COLAB = 'google.colab' in sys.modules
print('Colab:', IN_COLAB)

In [ ]:
# =====================================================================
# 1. CONFIGURAZIONE
# =====================================================================
class CFG:
    # --- modello ---------------------------------------------------------
    MODEL        = 'SamueleZurlo/Qwen2.5-3B-v3'   # deployment personale DeepInfra
    TOKENIZER_ID = 'Qwen/Qwen2.5-3B'              # tokenizer del modello generante
    BASE_URL     = 'https://api.deepinfra.com/v1/openai'
    CONTEXT_TOK  = 32000                          # Tab. 3 del paper: 32K per Qwen2.5-3B

    # --- griglia ---------------------------------------------------------
    TEMPERATURE  = [round(0.4 + 0.1 * i, 1) for i in range(12)]   # 0.4 ... 1.5

    # >>> L'UNICA MANOPOLA CHE CONTA <<<
    # numero di campioni per temperatura. Il notebook procede a GIRI: il giro j
    # genera un campione a ogni temperatura, da 0.4 a 1.5, poi ricomincia.
    # Interrompendo fra un giro e l'altro si hanno sempre celle equilibrate.
    N_SAMPLE     = 3

    # --- lunghezza -------------------------------------------------------
    TARGET_TOKENS       = 24000   # come il paper e come la tesi
    MIN_SUCCESS_TOKENS  = 20000   # soglia di successo: e' la lunghezza di analisi
    MAX_TOKENS_PER_REQ  = 24000   # il deployment lo accetta in una sola chiamata

    # --- campionamento: SOLA TEMPERATURA ---------------------------------
    TOP_P              = 1.0
    TOP_K              = -1
    REPETITION_PENALTY = 1.0
    FREQUENCY_PENALTY  = 0.0
    PRESENCE_PENALTY   = 0.0

    # --- robustezza ------------------------------------------------------
    CONCURRENCY        = 6    # documenti in volo dentro un giro
    MAX_ATTEMPTS       = 6    # tentativi per cella prima di arrendersi
    MAX_CONTINUAZIONI  = 60   # riprese massime dentro un singolo tentativo
    MAX_STALLI         = 8    # risposte vuote consecutive = tentativo bruciato
    TIMEOUT_S          = 900
    RETRY_HTTP         = 5    # ritentativi su errore di rete / 429 / 5xx

    # --- salvataggio e ripresa -------------------------------------------
    # Ogni tot token il documento in corso viene salvato in un checkpoint, cosi'
    # un'interruzione non butta via il lavoro gia' pagato. 0 disattiva.
    CHECKPOINT_OGNI = 4000

    # --- semi ------------------------------------------------------------
    SEED_TOKEN   = 20260827   # sceglie i token iniziali (uno per giro)
    SEED_BASE    = 700000     # base per i semi di campionamento

    # --- analisi ---------------------------------------------------------
    TOKENIZER_ANALISI = 'cl100k_base'   # identico alla tesi
    N_TOK_ANALISI     = 20000           # identico alla tesi

    # --- output ----------------------------------------------------------
    # Il prefisso e' diverso da quello dei cinque dataset dello sweep, cosi' il
    # notebook di analisi non lo raccoglie insieme a quelli.
    OUT_DIR   = Path('.')       # la cella 1.1 puo' spostarlo su Drive
    NOME_BASE = 'sweep_token_singolo_qwen2.5-3b'

CFG.OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_ID   = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
FILE_OK  = CFG.OUT_DIR / f'{CFG.NOME_BASE}.jsonl'
FILE_KO  = CFG.OUT_DIR / f'{CFG.NOME_BASE}_failed.jsonl'
DIR_CK   = CFG.OUT_DIR / f'{CFG.NOME_BASE}_checkpoint'
DIR_CK.mkdir(parents=True, exist_ok=True)

print(f'run_id            {RUN_ID}')
print(f'modello           {CFG.MODEL}')
print(f'temperature       {CFG.TEMPERATURE}')
print(f'campioni per T    {CFG.N_SAMPLE}   ->  {CFG.N_SAMPLE * len(CFG.TEMPERATURE)} documenti')
print(f'token per doc     {CFG.TARGET_TOKENS}  (successo >= {CFG.MIN_SUCCESS_TOKENS})')
print(f'output            {FILE_OK}')
print(f'checkpoint        {DIR_CK}')
print()
print('I file vengono SEMPRE aperti in aggiunta: una nuova run amplia il')
print('dataset esistente e non lo sovrascrive mai.')

## Montaggio di Drive (facoltativo)

Se vuoi che il `.jsonl` sopravviva alla sessione di Colab, esegui la cella
seguente **prima** di generare: monta Drive e sposta lì la cartella di output.
Altrimenti salta pure: alla fine c'è una cella che scarica il file sul computer.

In [ ]:
# =====================================================================
# 1.1 DRIVE  (facoltativo, solo su Colab)
# =====================================================================
USA_DRIVE = False        # <-- metti True per salvare su Drive

if USA_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CFG.OUT_DIR = Path('/content/drive/MyDrive/tesi_token_singolo')
    CFG.OUT_DIR.mkdir(parents=True, exist_ok=True)
    FILE_OK = CFG.OUT_DIR / f'{CFG.NOME_BASE}.jsonl'
    FILE_KO = CFG.OUT_DIR / f'{CFG.NOME_BASE}_failed.jsonl'
    DIR_CK  = CFG.OUT_DIR / f'{CFG.NOME_BASE}_checkpoint'
    DIR_CK.mkdir(parents=True, exist_ok=True)
    print('output ->', FILE_OK)
else:
    print('output locale ->', FILE_OK.resolve())

In [ ]:
# =====================================================================
# 2. CHIAVE E CLIENT
# =====================================================================
CHIAVE = None

# 1) segreto di Colab, se configurato (icona della chiave nella barra laterale)
if IN_COLAB:
    try:
        from google.colab import userdata
        CHIAVE = userdata.get('DEEPINFRA_API_KEY')
    except Exception:
        pass

# 2) variabile d'ambiente
CHIAVE = CHIAVE or os.environ.get('DEEPINFRA_API_KEY') or os.environ.get('DEEPINFRA_TOKEN')

# 3) inserimento a mano, non lasciato in chiaro nel notebook
if not CHIAVE:
    import getpass
    CHIAVE = getpass.getpass('DeepInfra API key: ').strip()

client = OpenAI(api_key=CHIAVE, base_url=CFG.BASE_URL, timeout=CFG.TIMEOUT_S, max_retries=0)
print('client pronto')

## Sonda dei parametri

Il modello è **base**, quindi si usa l'endpoint `completions` e non
`chat/completions`: quest'ultimo applicherebbe un template di chat e il prompt
non sarebbe più un token solo.

`top_k` e `repetition_penalty` non fanno parte dello schema OpenAI e viaggiano
in `extra_body`. La cella qui sotto verifica con una richiesta da 8 token quali
parametri il deployment accetta davvero, e memorizza il risultato: se qualcosa
viene rifiutato, viene tolto e la generazione parte comunque, ma il fatto resta
scritto nel dataset.

In [ ]:
# =====================================================================
# 2.1 SONDA: quali parametri accetta il deployment
# =====================================================================
EXTRA_BASE = {'top_k': CFG.TOP_K, 'repetition_penalty': CFG.REPETITION_PENALTY}
EXTRA_OK   = dict(EXTRA_BASE)
STD_OK     = {'frequency_penalty': CFG.FREQUENCY_PENALTY,
              'presence_penalty':  CFG.PRESENCE_PENALTY,
              'seed':              CFG.SEED_BASE}

def _prova(extra, std):
    r = client.completions.create(
        model=CFG.MODEL, prompt='The', max_tokens=8,
        temperature=1.0, top_p=CFG.TOP_P, extra_body=extra, **std)
    return r

# Prova il set completo; a ogni fallimento toglie un parametro e riprova.
# L'ordine di rimozione va dal meno al piu' importante da conservare.
passata = False
for step in [None, 'repetition_penalty', 'top_k', 'freq_pres', 'seed']:
    if step == 'repetition_penalty':
        EXTRA_OK.pop('repetition_penalty', None)
    elif step == 'top_k':
        EXTRA_OK.pop('top_k', None)
    elif step == 'freq_pres':
        STD_OK.pop('frequency_penalty', None)
        STD_OK.pop('presence_penalty', None)
    elif step == 'seed':
        STD_OK.pop('seed', None)
    try:
        r = _prova(EXTRA_OK, STD_OK)
        print('parametri accettati:', {**EXTRA_OK, **STD_OK})
        print('risposta di prova  :', repr(r.choices[0].text[:60]))
        print('usage              :', r.usage)
        passata = True
        break
    except Exception as e:
        etichetta = 'set completo' if step is None else f'senza {step}'
        print(f'{etichetta}: {type(e).__name__}: {str(e)[:160]}')

if not passata:
    raise SystemExit('il deployment non risponde con nessuna combinazione di parametri')

USA_SEED = 'seed' in STD_OK
PARAM_PERSI = sorted(set(EXTRA_BASE) - set(EXTRA_OK)) + \
              sorted({'frequency_penalty', 'presence_penalty', 'seed'} - set(STD_OK))
if PARAM_PERSI:
    print('\nATTENZIONE: parametri non supportati e quindi omessi:', PARAM_PERSI)
    print('Verranno registrati nel dataset come non applicati.')

## I token iniziali

«A single random, seed-controlled token». Un token va scelto in modo che
**sopravviva al giro di andata e ritorno**: lo estraiamo per id, lo decodifichiamo
in stringa, e la stringa viene rimandata al server che la ritokenizza. Se la
ritokenizzazione non restituisce lo stesso id, il prompt effettivo non è più
quel token, quindi quei candidati vanno scartati. Si escludono anche i token
speciali, quelli vuoti o di sola spaziatura e i frammenti di byte-fallback che
non formano UTF-8 valido.

Un token per giro, riusato a tutte e dodici le temperature.

In [ ]:
# =====================================================================
# 3. SCELTA DEI TOKEN INIZIALI
# =====================================================================
tok_modello = AutoTokenizer.from_pretrained(CFG.TOKENIZER_ID)
SPECIALI    = set(tok_modello.all_special_ids)
V           = tok_modello.vocab_size
print(f'vocabolario di {CFG.TOKENIZER_ID}: {V} voci, {len(SPECIALI)} speciali')

def token_valido(tid):
    """Il token deve decodificare in testo stampabile e ritokenizzare in se stesso."""
    if tid in SPECIALI:
        return None
    s = tok_modello.decode([tid])
    if not s or not s.strip():
        return None
    if '\ufffd' in s:                     # byte-fallback non decodificabile
        return None
    if tok_modello.encode(s, add_special_tokens=False) != [tid]:
        return None
    return s

rng_token = np.random.default_rng(CFG.SEED_TOKEN)
TOKEN_GIRO = []                              # [(id, stringa)] uno per giro
visti = set()
while len(TOKEN_GIRO) < CFG.N_SAMPLE:
    tid = int(rng_token.integers(0, V))
    if tid in visti:
        continue
    visti.add(tid)
    s = token_valido(tid)
    if s is not None:
        TOKEN_GIRO.append((tid, s))

print('\ntoken iniziali, uno per giro:')
for j, (tid, s) in enumerate(TOKEN_GIRO):
    print(f'  giro {j}:  id={tid:<7} {s!r}')

In [ ]:
# =====================================================================
# 4. GENERAZIONE DI UN DOCUMENTO
# =====================================================================
# DeepInfra ignora ignore_eos: quando il modello emette il token di fine
# sequenza la chiamata termina e va ripresa dal testo accumulato. E' la stessa
# 'generazione in continuazione' del par. 3.2 della tesi. Poiche' un modello
# autoregressivo non ha stato interno persistente, riprendere dal testo
# accumulato produce la stessa distribuzione condizionata.
#
# Lo stesso meccanismo rende recuperabile un'interruzione: il prompt da cui
# ripartire e' sempre  token_iniziale + testo_accumulato,  quindi basta
# conservare il testo accumulato per riprendere in una sessione successiva.

EOS_IDS = sorted({i for i in [tok_modello.eos_token_id,
                              tok_modello.convert_tokens_to_ids('<|im_end|>')]
                  if isinstance(i, int) and i >= 0})

# Segnale cooperativo di arresto: lo alza la cella 6 quando ricevi
# un'interruzione, e i cicli qui sotto lo guardano prima di ogni richiesta.
FERMATI = threading.Event()

class Interrotto(Exception):
    pass


# --------------------------- checkpoint --------------------------------
def _percorso_ck(T, sample_id):
    return DIR_CK / f'T{float(T):.1f}_g{int(sample_id)}.json'

def _salva_checkpoint(dati):
    """Scrittura atomica: prima su .tmp, poi rinomina. Un'interruzione a meta'
    non puo' quindi lasciare un checkpoint troncato."""
    p   = _percorso_ck(dati['temperature'], dati['sample_id'])
    tmp = p.with_suffix('.tmp')
    with open(tmp, 'w', encoding='utf-8') as fh:
        json.dump(dati, fh, ensure_ascii=False)
        fh.flush()
        os.fsync(fh.fileno())
    os.replace(tmp, p)

def _carica_checkpoint(T, sample_id, token_id):
    p = _percorso_ck(T, sample_id)
    if not p.exists():
        return None
    try:
        with open(p, encoding='utf-8') as fh:
            d = json.load(fh)
    except Exception as e:
        print(f'  checkpoint illeggibile {p.name}: {e} — lo ignoro')
        return None
    # il token iniziale deve essere quello giusto, altrimenti ripartire da
    # quel testo significherebbe mescolare due documenti diversi
    if int(d.get('prompt_token_id', -1)) != int(token_id):
        print(f'  checkpoint {p.name} appartiene a un altro token — lo ignoro')
        return None
    if int(d.get('completion_tokens', 0)) <= 0:
        return None
    return d

def _elimina_checkpoint(T, sample_id):
    for p in (_percorso_ck(T, sample_id), _percorso_ck(T, sample_id).with_suffix('.tmp')):
        try:
            p.unlink()
        except FileNotFoundError:
            pass


# --------------------------- una richiesta -----------------------------
def _chiamata(prompt, max_tok, T, seed):
    """Una richiesta, con ritentativi su errori transitori."""
    std = dict(STD_OK)
    if 'seed' in std:
        std['seed'] = int(seed) % (2**31 - 1)
    ultimo = None
    for k in range(CFG.RETRY_HTTP):
        if FERMATI.is_set():
            raise Interrotto('arresto richiesto')
        try:
            r = client.completions.create(
                model=CFG.MODEL, prompt=prompt, max_tokens=int(max_tok),
                temperature=float(T), top_p=CFG.TOP_P,
                extra_body=EXTRA_OK, **std)
            ch  = r.choices[0]
            usa = getattr(r, 'usage', None)
            return dict(text=ch.text or '',
                        finish=ch.finish_reason,
                        n_out=int(getattr(usa, 'completion_tokens', 0) or 0),
                        n_in=int(getattr(usa, 'prompt_tokens', 0) or 0))
        except Interrotto:
            raise
        except Exception as e:
            ultimo = e
            time.sleep(min(2 ** k, 30) + random.random())
    raise Interrotto(f'{type(ultimo).__name__}: {str(ultimo)[:200]}')


# --------------------------- un documento ------------------------------
def genera_documento(T, sample_id, token_id, token_str, tentativo):
    """Un tentativo completo per una cella (T, sample_id).

    Se esiste un checkpoint per quella cella, riparte da li' invece che da zero.
    Se arriva un'interruzione, salva quanto fatto e restituisce un record
    marcato interrotto=True."""
    t0    = time.time()
    seed0 = CFG.SEED_BASE + int(round(T * 10)) * 1000 + sample_id * 10 + tentativo

    testo, n_tok, n_in, n_cont = '', 0, 0, 0
    eos_positions, secondi_prima = [], 0.0

    ck = _carica_checkpoint(T, sample_id, token_id)
    if ck:
        testo         = ck['generated_text']
        n_tok         = int(ck['completion_tokens'])
        n_cont        = int(ck.get('n_continuations', 0))
        n_in          = int(ck.get('api_prompt_tokens', 0))
        eos_positions = list(ck.get('eos_positions', []))
        secondi_prima = float(ck.get('generation_seconds', 0.0))
        print(f'  ripresa T={T} giro {sample_id}: riparto da {n_tok} token '
              f'(mancano {CFG.TARGET_TOKENS - n_tok})')

    ultimo_ck = n_tok
    stalli    = 0
    stalled   = False
    interrotto = False
    finish    = None
    errore    = None

    def _istantanea():
        return dict(temperature=float(T), sample_id=int(sample_id),
                    prompt_token_id=int(token_id), prompt_token_str=token_str,
                    generated_text=testo, completion_tokens=int(n_tok),
                    n_continuations=int(n_cont), eos_positions=eos_positions,
                    api_prompt_tokens=int(n_in),
                    generation_seconds=round(secondi_prima + time.time() - t0, 1),
                    salvato_il=datetime.datetime.now(datetime.timezone.utc).isoformat())

    try:
        while n_tok < CFG.TARGET_TOKENS:
            if FERMATI.is_set():
                interrotto = True
                break

            resta  = CFG.TARGET_TOKENS - n_tok
            chunk  = min(CFG.MAX_TOKENS_PER_REQ, resta)
            prompt = token_str + testo

            # guardia sul context window: prompt + richiesta devono starci
            n_prompt_stimato = 1 + n_tok
            if n_prompt_stimato + chunk > CFG.CONTEXT_TOK:
                chunk = max(0, CFG.CONTEXT_TOK - n_prompt_stimato - 8)
                if chunk <= 0:
                    stalled = True
                    break

            r = _chiamata(prompt, chunk, T, seed0 + n_cont)
            n_in += r['n_in']

            if r['n_out'] == 0 or not r['text']:
                # ripresa a vuoto: il modello rimette subito EOS
                stalli += 1
                n_cont += 1
                if stalli >= CFG.MAX_STALLI or n_cont > CFG.MAX_CONTINUAZIONI:
                    stalled = True
                    break
                continue

            stalli  = 0
            testo  += r['text']
            n_tok  += r['n_out']
            finish  = r['finish']

            if r['finish'] == 'stop':
                eos_positions.append(n_tok)

            # checkpoint periodico: quello che e' stato pagato resta su disco
            if CFG.CHECKPOINT_OGNI and (n_tok - ultimo_ck) >= CFG.CHECKPOINT_OGNI \
                    and n_tok < CFG.TARGET_TOKENS:
                _salva_checkpoint(_istantanea())
                ultimo_ck = n_tok

            if n_tok < CFG.TARGET_TOKENS:
                n_cont += 1
                if n_cont > CFG.MAX_CONTINUAZIONI:
                    stalled = True
                    break

    except Interrotto as e:
        errore = str(e)
        interrotto = FERMATI.is_set()

    ok = (n_tok >= CFG.MIN_SUCCESS_TOKENS) and not interrotto

    # Il checkpoint sopravvive solo se il documento NON e' finito: se e' andato
    # a buon fine il record completo e' nel jsonl e il checkpoint e' zavorra.
    if ok:
        _elimina_checkpoint(T, sample_id)
    elif n_tok > 0:
        _salva_checkpoint(_istantanea())

    return dict(
        # --- schema identico ai dataset della tesi -----------------------
        model=CFG.MODEL,
        temperature=float(T),
        sample_id=int(sample_id),
        prompt_length_tokens=1,
        prompt_length_chars=len(token_str),
        prompt_sha256_16=hashlib.sha256(token_str.encode('utf-8')).hexdigest()[:16],
        generated_text=testo,
        finish_reason=finish,
        completion_tokens=int(n_tok),
        generated_chars=len(testo),
        success=bool(ok),
        min_success_tokens=CFG.MIN_SUCCESS_TOKENS,
        generation_mode='continuation',
        logit_bias_applied=None,
        stop_token_ids=EOS_IDS,
        n_continuations=int(n_cont),
        eos_positions=eos_positions,
        stalled=bool(stalled),
        max_tokens=CFG.TARGET_TOKENS,
        top_p=CFG.TOP_P,
        top_k=EXTRA_OK.get('top_k'),
        repetition_penalty=EXTRA_OK.get('repetition_penalty'),
        frequency_penalty=STD_OK.get('frequency_penalty'),
        presence_penalty=STD_OK.get('presence_penalty'),
        seed=int(seed0),
        top_k_verified=None,
        run_id=RUN_ID,
        attempt=int(tentativo),
        is_retry=bool(tentativo > 0),
        concurrency=CFG.CONCURRENCY,
        generated_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        generation_seconds=round(secondi_prima + time.time() - t0, 1),
        tokenizer_id=CFG.TOKENIZER_ID,
        # --- campi nuovi di questa run -----------------------------------
        prompt_mode='single_token',
        prompt_token_id=int(token_id),
        prompt_token_str=token_str,
        round_index=int(sample_id),
        api_prompt_tokens=int(n_in),
        parametri_non_supportati=PARAM_PERSI,
        interrotto=bool(interrotto),
        ripreso_da_checkpoint=bool(ck is not None),
        errore=errore,
    )

print('EOS ids:', EOS_IDS)

## Lo scheduler a giri

Ogni giro lancia le dodici temperature in parallelo, in ordine crescente di
sottomissione, e **attende che il giro sia completo** prima di passare al
successivo. Dentro ogni cella si ritenta con un seme diverso — ma **con lo stesso
token iniziale**, altrimenti l'appaiamento fra temperature si romperebbe —
finché non si arriva ad almeno un successo o si esauriscono i tentativi.

È attorno a $T \simeq 1$ che i tentativi servono: è la regione in cui il modello
emette EOS di continuo e un documento può fermarsi prima dei 20 000 token.

In [ ]:
# =====================================================================
# 5. SCHEDULER A GIRI
# =====================================================================
_lock = threading.Lock()

def _scrivi(rec, file):
    """Aggiunge una riga al file, senza mai troncarlo, e forza la scrittura
    su disco: un'interruzione subito dopo non puo' perdere il record."""
    with _lock:
        with open(file, 'a', encoding='utf-8') as fh:
            fh.write(json.dumps(rec, ensure_ascii=False) + '\n')
            fh.flush()
            os.fsync(fh.fileno())


def stato(stampa=True):
    """Che cosa c'e' gia' su disco. Legge i file, non la memoria, cosi' funziona
    anche dopo un riavvio del runtime."""
    fatte = collections.defaultdict(set)
    if FILE_OK.exists():
        for riga in open(FILE_OK, encoding='utf-8'):
            try:
                d = json.loads(riga)
            except Exception:
                continue
            if d.get('success'):
                fatte[round(float(d['temperature']), 1)].add(int(d['sample_id']))

    parziali = {}
    for p in sorted(DIR_CK.glob('*.json')):
        try:
            with open(p, encoding='utf-8') as fh:
                d = json.load(fh)
            parziali[(round(float(d['temperature']), 1), int(d['sample_id']))] = \
                int(d['completion_tokens'])
        except Exception:
            continue

    if stampa:
        n_fatte = sum(len(v) for v in fatte.values())
        attese  = CFG.N_SAMPLE * len(CFG.TEMPERATURE)
        print(f'{n_fatte}/{attese} celle complete su disco'
              + (f', {len(parziali)} parziali da riprendere' if parziali else ''))
        print(f'{"T":>5}  {"giri completi":<22} {"parziale"}')
        for T in CFG.TEMPERATURE:
            g = sorted(fatte.get(T, ()))
            pz = [f'giro {j}: {n} token' for (TT, j), n in sorted(parziali.items()) if TT == T]
            print(f'{T:>5.1f}  {str(g) if g else "-":<22} {", ".join(pz) if pz else ""}')
    return fatte, parziali


def cella(T, sample_id, token_id, token_str):
    """Ritenta finche' la cella non produce almeno un successo."""
    rec = None
    for tent in range(CFG.MAX_ATTEMPTS):
        if FERMATI.is_set():
            break
        rec = genera_documento(T, sample_id, token_id, token_str, tent)
        if rec['success']:
            _scrivi(rec, FILE_OK)
            return rec
        if rec['interrotto']:
            # non e' un fallimento del modello: il lavoro e' nel checkpoint
            print(f'    T={T:<4} giro {sample_id}: interrotto a '
                  f'{rec["completion_tokens"]} token, salvato nel checkpoint')
            return rec
        _scrivi(rec, FILE_KO)
        print(f'    T={T:<4} giro {sample_id}  tentativo {tent}: '
              f'{rec["completion_tokens"]} token, stalled={rec["stalled"]}'
              + (f', errore={rec["errore"]}' if rec['errore'] else '') + ' -> ritento')
    return rec


def esegui(giri=None, temperature=None):
    giri        = range(CFG.N_SAMPLE) if giri is None else giri
    temperature = CFG.TEMPERATURE if temperature is None else temperature
    FERMATI.clear()
    t_avvio = time.time()
    interrotta = False

    for j in giri:
        if FERMATI.is_set():
            break
        token_id, token_str = TOKEN_GIRO[j]
        print(f'\n=== GIRO {j}  -  token {token_id} {token_str!r} '
              f'-  {len(temperature)} temperature ===')
        t_giro = time.time()
        fatti  = []

        ex = ThreadPoolExecutor(max_workers=CFG.CONCURRENCY)
        futuri = {}
        try:
            for T in temperature:                 # sottomissione in ordine crescente
                futuri[ex.submit(cella, T, j, token_id, token_str)] = T
            for f in as_completed(futuri):
                T = futuri[f]
                try:
                    rec = f.result()
                except Exception as e:
                    print(f'  T={T}: eccezione non gestita {type(e).__name__}: {e}')
                    continue
                if rec is None:
                    continue
                fatti.append(rec)
                stato_txt = 'ok ' if rec['success'] else ('INT' if rec.get('interrotto') else 'KO ')
                print(f'  {stato_txt} T={T:<4} {rec["completion_tokens"]:>6} token, '
                      f'{rec["n_continuations"]:>3} riprese, '
                      f'{len(rec["eos_positions"]):>3} EOS, '
                      f'{rec["generation_seconds"]:>6.1f}s')
        except KeyboardInterrupt:
            interrotta = True
            FERMATI.set()
            print('\n*** INTERRUZIONE RICHIESTA ***')
            print('Annullo il lavoro non ancora partito e aspetto che i documenti')
            print('in volo si chiudano e salvino il checkpoint. Puo\' volerci fino a')
            print('una richiesta (max_tokens=%d). Non serve interrompere di nuovo:'
                  % CFG.MAX_TOKENS_PER_REQ)
            print('quello che e\' stato generato e\' gia\' al sicuro su disco.')
            for f in futuri:
                f.cancel()
        finally:
            ex.shutdown(wait=True)

        ok = sum(1 for r in fatti if r.get('success'))
        print(f'--- giro {j} chiuso in {time.time()-t_giro:.0f}s: '
              f'{ok}/{len(temperature)} celle riuscite ---')
        if FERMATI.is_set():
            break

    print('\n########  ' + ('INTERROTTA' if interrotta else 'FINE') + '  ########')
    print(f'tempo totale        {time.time()-t_avvio:.0f}s')
    print(f'file                {FILE_OK}')
    print()
    stato()
    if interrotta:
        print('\nPer riprendere: riesegui la cella 6. Le celle complete vengono')
        print('saltate e quelle parziali ripartono dal punto in cui erano.')

print('scheduler pronto. Lancia la cella successiva.')

In [ ]:
# =====================================================================
# 6. ESECUZIONE   (rieseguibile: riprende, non ricomincia)
# =====================================================================
# Controllo di compatibilita': se il file esiste gia', deve essere stato
# prodotto dallo stesso modello e con gli stessi token iniziali, altrimenti
# ampliarlo significherebbe mescolare due esperimenti diversi.
if FILE_OK.exists():
    modelli, token_visti = set(), {}
    for riga in open(FILE_OK, encoding='utf-8'):
        try:
            d = json.loads(riga)
        except Exception:
            continue
        modelli.add(d.get('model'))
        if d.get('sample_id') is not None and d.get('prompt_token_id') is not None:
            token_visti.setdefault(int(d['sample_id']), set()).add(int(d['prompt_token_id']))

    problemi = []
    if modelli - {CFG.MODEL}:
        problemi.append(f'il file contiene anche i modelli {modelli - {CFG.MODEL}}')
    for j, ids in sorted(token_visti.items()):
        if j < len(TOKEN_GIRO) and ids != {TOKEN_GIRO[j][0]}:
            problemi.append(f'il giro {j} sul disco usa il token {ids}, '
                            f'qui vale {TOKEN_GIRO[j][0]} (SEED_TOKEN cambiato?)')
    if problemi:
        print('ATTENZIONE, il file esistente non e\' compatibile:')
        for p in problemi:
            print('  -', p)
        print('Cambia CFG.NOME_BASE per scrivere su un file nuovo, oppure')
        print('rimetti i parametri di prima. Non proseguo.')
        raise SystemExit(1)
    print('file esistente compatibile: verra\' ampliato, non sovrascritto.\n')

print('STATO PRIMA DI PARTIRE')
print('-' * 60)
fatte, parziali = stato()

# le celle gia' complete vengono saltate; quelle parziali NO, riprendono
gia_fatte = {(T, j) for T, giri in fatte.items() for j in giri}

# Il filtro viene applicato UNA VOLTA SOLA: rieseguendo questa cella senza la
# guardia si avvolgerebbe il wrapper in se stesso e si andrebbe in ricorsione.
if not globals().get('_CELLA_AVVOLTA', False):
    _cella_originale = cella

    def cella(T, sample_id, token_id, token_str):
        if (round(float(T), 1), int(sample_id)) in gia_fatte:
            return dict(completion_tokens=0, api_prompt_tokens=0, success=True,
                        n_continuations=0, eos_positions=[], generation_seconds=0.0,
                        interrotto=False, saltata=True)
        return _cella_originale(T, sample_id, token_id, token_str)

    _CELLA_AVVOLTA = True

print('\nGENERAZIONE')
print('-' * 60)
esegui()

# Analisi

$R$ viene calcolato **esattamente come nel notebook della tesi**: `cl100k_base`,
troncamento a 20 000 token di analisi,
$R = |\mathcal{D}_2 \setminus \mathcal{D}_1| / |\mathcal{D}_1|$.

La curva col prompt lungo è quella del Capitolo 4, riportata qui come costante
così che il confronto funzioni anche senza avere il vecchio `.jsonl` a portata
di mano. Il riferimento umano è **0.507**, la baseline della Tabella 3.5 misurata
sugli stessi 20 000 token: lo 0.17 del paper è misurato su libri interi e a
questa lunghezza non è un termine di paragone valido (§2.3.4 della tesi).

In [ ]:
# =====================================================================
# 7. R  —  stessa procedura del notebook di analisi della tesi
# =====================================================================
enc = tiktoken.get_encoding(CFG.TOKENIZER_ANALISI)

def tokenizza(testo, n=None):
    t = enc.encode(testo, disallowed_special=())
    return t[:(n or CFG.N_TOK_ANALISI)]

def descrittore_R(tokens):
    """[Z] sec.4.2: |V2 \\ V1| / |V1|."""
    if len(tokens) < 20:
        return float('nan')
    h = len(tokens) // 2
    V1, V2 = set(tokens[:h]), set(tokens[h:])
    return len(V2 - V1) / len(V1) if V1 else float('nan')

def ttr(tokens):
    return len(set(tokens)) / len(tokens) if len(tokens) else float('nan')

# --- carica la run nuova -------------------------------------------------
righe = [json.loads(r) for r in open(FILE_OK, encoding='utf-8')] if FILE_OK.exists() else []
righe = [d for d in righe if d.get('success') and d.get('generated_text')]
print(f'{len(righe)} documenti riusciti')

per_T = collections.defaultdict(list)
per_T_ttr = collections.defaultdict(list)
per_T_full = collections.defaultdict(list)      # convenzione del paper: testo intero
for d in righe:
    tk = tokenizza(d['generated_text'])
    if len(tk) < CFG.N_TOK_ANALISI:
        print(f"  T={d['temperature']} giro {d['sample_id']}: solo {len(tk)} token di analisi")
    per_T[round(float(d['temperature']), 1)].append(descrittore_R(tk))
    per_T_ttr[round(float(d['temperature']), 1)].append(ttr(tk))
    per_T_full[round(float(d['temperature']), 1)].append(
        descrittore_R(enc.encode(d['generated_text'], disallowed_special=())))

Ts    = sorted(per_T)
R_new = np.array([np.mean(per_T[T]) for T in Ts])
R_sd  = np.array([np.std(per_T[T], ddof=1) if len(per_T[T]) > 1 else 0.0 for T in Ts])
R_n   = np.array([len(per_T[T]) for T in Ts])

# --- la curva col prompt lungo, dal Capitolo 4 ---------------------------
R_LUNGO = {   # T: (media, dev.std, n)   Qwen2.5-3B, prompt di 2000 token
    0.4: (0.002793, 0.003863, 5), 0.5: (0.009679, 0.016871, 5),
    0.6: (0.000000, 0.000000, 5), 0.7: (0.002226, 0.002086, 5),
    0.8: (0.011157, 0.024948, 5), 0.9: (0.003198, 0.002180, 3),
    1.0: (0.455170, 0.000000, 1), 1.1: (0.537791, 0.066111, 4),
    1.2: (0.619269, 0.038008, 5), 1.3: (0.658088, 0.018494, 5),
    1.4: (0.688326, 0.029988, 5), 1.5: (0.688285, 0.006592, 5),
}
R_UMANO = 0.507      # Tab. 3.5, misurata su 20 000 token di Guerra e pace

print(f'\n{"T":>5} {"R token singolo":>18} {"sd":>7} {"n":>3}   {"R prompt lungo":>15}')
for i, T in enumerate(Ts):
    lu = R_LUNGO.get(T, (float("nan"),) * 3)[0]
    print(f'{T:>5.1f} {R_new[i]:>18.4f} {R_sd[i]:>7.4f} {R_n[i]:>3d}   {lu:>15.4f}')

imax = int(np.nanargmax(R_new))
print(f'\nmassimo della curva a token singolo: T = {Ts[imax]}  (R = {R_new[imax]:.4f})')
print('la curva col prompt lungo è monotona crescente fino a T = 1.5')

In [ ]:
# =====================================================================
# 8. LA FIGURA
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))

# --- pannello A: le due curve -------------------------------------------
ax = axes[0]
Tl  = sorted(R_LUNGO)
ml  = [R_LUNGO[T][0] for T in Tl]
sl  = [R_LUNGO[T][1] for T in Tl]
ax.errorbar(Tl, ml, yerr=sl, marker='s', ms=5, lw=1.6, capsize=3,
            color='#d95f02', label='prompt 2000 token (Cap. 4)')
ax.errorbar(Ts, R_new, yerr=R_sd, marker='o', ms=5, lw=1.6, capsize=3,
            color='#1b9e77', label='token singolo (questa run)')
ax.axhline(R_UMANO, color='k', ls='--', lw=1.2,
           label=f'baseline umana {R_UMANO} (20k token)')
ax.set_xlabel('temperatura $T$')
ax.set_ylabel('$R = |\\mathcal{D}_2 \\setminus \\mathcal{D}_1| / |\\mathcal{D}_1|$')
ax.set_title('A — il descrittore $R$ contro la temperatura')
ax.legend(loc='upper left', fontsize=8.5)

# --- pannello B: differenza ---------------------------------------------
ax = axes[1]
comuni = [T for T in Ts if T in R_LUNGO]
diff   = [R_new[Ts.index(T)] - R_LUNGO[T][0] for T in comuni]
ax.axhline(0, color='k', lw=1)
ax.bar(comuni, diff, width=0.07, color=['#1b9e77' if d > 0 else '#d95f02' for d in diff])
ax.set_xlabel('temperatura $T$')
ax.set_ylabel('$R$(token singolo) $-$ $R$(prompt lungo)')
ax.set_title('B — dove il prompt sposta $R$')

plt.tight_layout()
for ext in ('png', 'pdf'):
    plt.savefig(CFG.OUT_DIR / f'confronto_prompt_R.{ext}')
plt.show()

# --- verdetto ------------------------------------------------------------
print('VERDETTO')
print('-' * 66)
monotona = all(R_new[i] <= R_new[i+1] + 1e-9 for i in range(len(R_new) - 1))
if monotona:
    print('La curva a token singolo è anch\'essa monotona: il massimo NON compare,')
    print('e la spiegazione basata sul prompt NON è confermata da questi dati.')
else:
    coda = [R_new[i] for i, T in enumerate(Ts) if T > Ts[imax]]
    calo = (R_new[imax] - min(coda)) / R_new[imax] if coda and R_new[imax] > 0 else 0.0
    print(f'Massimo a T = {Ts[imax]} con R = {R_new[imax]:.3f}, poi un calo del {100*calo:.0f}%')
    print('verso le temperature alte. La curva col prompt lungo, sullo stesso')
    print('modello e con ogni altra condizione identica, sale invece in modo')
    print('monotono fino a 1.5. La differenza è attribuibile al solo prompt.')

In [ ]:
# =====================================================================
# 9. DIAGNOSTICA DELLA RUN
# =====================================================================
print('Riprese e stalli per temperatura (il costo vero della run)\n')
print(f'{"T":>5} {"n":>3} {"riprese med":>12} {"EOS med":>9} {"secondi med":>12} {"token prompt":>14}')
d_cont = collections.defaultdict(list)
for d in righe:
    d_cont[round(float(d['temperature']), 1)].append(d)
for T in sorted(d_cont):
    g = d_cont[T]
    print(f'{T:>5.1f} {len(g):>3} '
          f'{np.mean([x["n_continuations"] for x in g]):>12.1f} '
          f'{np.mean([len(x["eos_positions"]) for x in g]):>9.1f} '
          f'{np.mean([x["generation_seconds"] for x in g]):>12.1f} '
          f'{np.mean([x["api_prompt_tokens"] for x in g]):>14,.0f}')

if FILE_KO.exists():
    ko = [json.loads(r) for r in open(FILE_KO, encoding='utf-8')]
    print(f'\n{len(ko)} tentativi falliti, per temperatura:')
    c = collections.Counter(round(float(d['temperature']), 1) for d in ko)
    for T in sorted(c):
        print(f'  T={T}: {c[T]}')
else:
    print('\nnessun tentativo fallito')

print('\nToken iniziali usati:')
for j, (tid, s) in enumerate(TOKEN_GIRO):
    print(f'  giro {j}: id={tid} {s!r}')

In [ ]:
# =====================================================================
# 10. SCARICA I RISULTATI
# =====================================================================
if IN_COLAB and not USA_DRIVE:
    from google.colab import files
    for f in [FILE_OK, FILE_KO,
              CFG.OUT_DIR / 'confronto_prompt_R.png',
              CFG.OUT_DIR / 'confronto_prompt_R.pdf']:
        if Path(f).exists():
            files.download(str(f))
else:
    print('file in', CFG.OUT_DIR.resolve())
    for f in sorted(CFG.OUT_DIR.glob('*')):
        print('  ', f.name, f'{f.stat().st_size/1e6:.2f} MB')